# 02 — Transformasi ETL dengan PySpark

Notebook ini menjalankan alur **Extract → Transform → Load** pada data transaksi penuh. Jalankan sel secara berurutan. Proses pertama dapat memakan waktu beberapa menit karena Spark membaca hampir 10 juta baris.

In [1]:
import os

os.environ["JAVA_HOME"] = (
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 4g pyspark-shell"
)

In [2]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((
    folder for folder in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (folder / 'src').is_dir() and (folder / 'README.md').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Root proyek tidak ditemukan. Jalankan Jupyter dari folder repository.')
sys.path.insert(0, str(PROJECT_ROOT))

from pyspark import StorageLevel
from src.config import get_paths, validate_source_files
from src.spark_pipeline import (
    aggregate_outputs, create_spark, read_sources, transform_sources,
    validate_sources, write_csv_folder,
)

VARIANT = 'retail_contaminated_dataset'
paths = get_paths(VARIANT)
validate_source_files(paths)
paths

ProjectPaths(project_root=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/big-data-retail-analytics'), dataset_root=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/Synthetic Retail Dataset - 10 Million Transactions'), clean_dir=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/Synthetic Retail Dataset - 10 Million Transactions/retail_clean_dataset'), contaminated_dir=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/Synthetic Retail Dataset - 10 Million Transactions/retail_contaminated_dataset'), source_dir=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/Synthetic Retail Dataset - 10 Million Transactions/retail_contaminated_dataset'), output_dir=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/big-data-retail-analytics/output'), parquet_dir=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/big-data-retail-analytics/output/parquet/sales_enriched'), charts_dir=WindowsPath('D:/Latihan_Aiman/S5/560_BIG DATA/Project UTS/big-data

## 1. Extract
Spark membaca CSV memakai skema yang sudah ditentukan. Ini lebih aman dan lebih cepat daripada menebak tipe kolom setiap kali.

In [3]:
spark = create_spark('local[2]')
frames = read_sources(spark, paths.source_dir)
{name: frame.columns for name, frame in frames.items()}

{'sales': ['date',
  'receipt_id',
  'store_id',
  'sku_id',
  'customer_id',
  'quantity',
  'unit_price',
  'total_value',
  'channel',
  'discount_pct',
  'promo_id'],
 'stores': ['store_id', 'store_name', 'city', 'store_type', 'opening_date'],
 'skus': ['sku_id',
  'sku_name',
  'category',
  'subcategory',
  'unit_price',
  'cost_price',
  'brand'],
 'customers': ['cust_id',
  'age',
  'gender',
  'city',
  'loyalty_segment',
  'preferred_channel',
  'registration_date'],
 'promotions': ['promo_id',
  'promo_name',
  'start_date',
  'end_date',
  'discount_pct',
  'promo_type',
  'target_type',
  'target_value']}

## 2. Validasi sumber
Pemeriksaan ini mencari nilai wajib yang kosong, angka tidak wajar, total transaksi yang tidak cocok, serta ID yang tidak mempunyai pasangan pada tabel master.

In [4]:
checks = validate_sources(frames)
checks

{'input_rows': 9945511,
 'rows_with_required_nulls': 0,
 'invalid_quantity_rows': 0,
 'invalid_unit_price_rows': 0,
 'invalid_discount_rows': 0,
 'formula_mismatch_rows': 0,
 'orphan_store_rows': 0,
 'orphan_sku_rows': 0,
 'orphan_customer_rows': 0,
 'orphan_promo_rows': 0}

## 3. Transform
Transaksi digabungkan dengan data toko, produk, pelanggan, dan promosi. Kolom analisis seperti bulan, penjualan kotor, diskon, laba kotor, margin, dan penanda promosi lalu dibuat.

In [5]:
enriched = transform_sources(frames).persist(StorageLevel.DISK_ONLY)
output_rows = enriched.count()
assert output_rows == checks['input_rows'], 'Jumlah baris berubah setelah transformasi'
assert all(value == 0 for key, value in checks.items() if key != 'input_rows'), checks
print(f'Baris hasil: {output_rows:,}')
enriched.select(
    'date', 'receipt_id', 'store_name', 'sku_name', 'category',
    'quantity', 'total_value', 'gross_profit', 'channel', 'is_promoted'
).show(10, truncate=False)

Baris hasil: 9,945,511
+----------+------------+-------------------------------+--------------------------------+-------------------------+--------+-----------+------------+--------+-----------+
|date      |receipt_id  |store_name                     |sku_name                        |category                 |quantity|total_value|gross_profit|channel |is_promoted|
+----------+------------+-------------------------------+--------------------------------+-------------------------+--------+-----------+------------+--------+-----------+
|2025-04-02|RCPT00000001|Lahore Convenience Store #16   |GreenValley Men's Wear Small    |Apparel & Footwear       |1       |2379.11    |751.2       |In-Store|false      |
|2025-04-02|RCPT00000001|Lahore Convenience Store #16   |ValueChoice Eggs Pack of 6      |Dairy & Bakery           |3       |1006.65    |278.28      |In-Store|false      |
|2022-04-24|RCPT00000002|Islamabad Convenience Store #15|NatureBlend Ready Meals 500ml   |Frozen Foods             |1

In [6]:
aggregates = aggregate_outputs(enriched)
aggregates['monthly_sales'].show(12, truncate=False)
aggregates['store_sales'].show(10, truncate=False)
aggregates['category_sales'].show(10, truncate=False)
aggregates['channel_sales'].show(truncate=False)
aggregates['promotion_comparison'].show(truncate=False)

+----------+----------+--------+----------+--------------+-------------+
|month     |line_items|receipts|units_sold|sales_value   |gross_profit |
+----------+----------+--------+----------+--------------+-------------+
|2022-01-01|159202    |82237   |299649    |1.8203952592E8|5.602258551E7|
|2022-02-01|149106    |77265   |279983    |1.7104354372E8|5.289897926E7|
|2022-03-01|176648    |92096   |332417    |2.0428331603E8|6.400130284E7|
|2022-04-01|175651    |91175   |330560    |2.0500127287E8|6.587783131E7|
|2022-05-01|186910    |96624   |351452    |2.0632851819E8|5.836146961E7|
|2022-06-01|187283    |97104   |352922    |1.6309772016E8|1.387671585E7|
|2022-07-01|177208    |91922   |333342    |2.0532459852E8|6.469420584E7|
|2022-08-01|195802    |101142  |367973    |2.2562070399E8|7.037680933E7|
|2022-09-01|195734    |101328  |367848    |2.2334216335E8|6.862663333E7|
|2022-10-01|197228    |101968  |370513    |2.3073812513E8|7.383519908E7|
|2022-11-01|231812    |119941  |435675    |2.686736

## 4. Load
Hasil rinci disimpan sebagai Parquet yang dipartisi per tahun dan bulan. Ringkasan kecil disimpan sebagai CSV agar mudah dibuka atau dimuat ke PostgreSQL.

In [7]:
paths.parquet_dir.parent.mkdir(parents=True, exist_ok=True)
paths.validation_dir.mkdir(parents=True, exist_ok=True)

enriched.write.mode('overwrite').partitionBy('year', 'month_key').parquet(
    str(paths.parquet_dir)
)
for name, frame in aggregates.items():
    write_csv_folder(frame, paths.validation_dir / name)

print('Parquet:', paths.parquet_dir)
print('Ringkasan CSV:', paths.validation_dir)

Py4JJavaError: An error occurred while calling o378.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:286)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.j

## 5. Verifikasi hasil load
Data dibaca ulang dari Parquet untuk memastikan jumlah baris tetap sama dan format hasil benar-benar dapat digunakan.

In [ ]:
reloaded = spark.read.parquet(str(paths.parquet_dir))
reloaded_rows = reloaded.count()
assert reloaded_rows == output_rows
print(f'Verifikasi berhasil: {reloaded_rows:,} baris')
reloaded.printSchema()

In [ ]:
enriched.unpersist()
spark.stop()